# Redis order-book cache

Fetch one Polymarket market's YES/NO order books, write top-of-book to Redis, and time each step.

Needs the `redis` service from `docker-compose.yml` (redis-stack, for RedisJSON).

In [1]:
import redis
import requests
from datetime import datetime
from typing import List
import time
import aiohttp
import asyncio


from src.PolyMarket import PolyMarket
from src.PolyDirection import PolyDirection
from src.PolyOrderBook import PolyOrderBook

## Async fetch
Fetch YES and NO books concurrently (`asyncio.gather`) over one shared `aiohttp` session.

In [2]:
async def async_get_ob(market: PolyMarket):
    async with aiohttp.ClientSession() as session:
        # Pass the same session throughout the lifecycle
        order_book_yes, order_book_no = await market.get_ob_async(session)

    return order_book_yes, order_book_no

## Connect to Redis
`ping()` returns `True` if the server is up.

In [3]:
try:
    r = redis.Redis(host="localhost", port=6379, db=0)
    print(r.ping())
except Exception as e:
    print(f"  Redis connection failed: {e}")

True


## Write helpers

| Helper | Key | Type | TTL | Holds |
|---|---|---|---|---|
| `set_redis_order_book` | `<slug>:<0\|1>` | JSON | 5 min | latest snapshot per direction |
| `push_redis_list` | `<slug>` | list | 2 h | snapshot history, newest first |

In [4]:
def set_redis_order_book(order_book: PolyOrderBook) -> None:
    # one key per market + direction (1 = YES, 0 = NO); overwritten on each call
    key = f"{order_book.market_slug}:{int(bool(order_book.direction))}"
    value = {
        "timestamp": order_book.timestamp,
        "highest_bid": order_book.bids_highest.get("price", 0),
        "lowest_ask": order_book.asks_lowest.get("price", 0),
    }
    r.json().set(key, "$", value)  # RedisJSON: store as a JSON doc at root
    r.expire(key, 300)  # drop stale snapshots after 5 min


def push_redis_list(list_order_book: List[PolyOrderBook]) -> None:
    for ob in list_order_book:
        # LPUSH -> newest at index 0; both directions share one list
        r.lpush(
            ob.market_slug,
            str(  # Python repr, not JSON: read back with ast.literal_eval
                {
                    "direction": int(bool(ob.direction)),
                    "timestamp": ob.timestamp,
                    "highest_bid": ob.bids_highest.get("price", 0),
                    "lowest_ask": ob.asks_lowest.get("price", 0),
                }
            ),
        )
        r.expire(ob.market_slug, 120 * 60)  # TTL resets on every push

## Run
Load market metadata, then fetch, write to Redis, and time each step.

In [6]:
request_session = requests.Session()

# market metadata (token ids etc.) via sync HTTP
pm = PolyMarket()
next_15m = int(((datetime.now().timestamp() // 900) + 1) * 900)
pm.slug = f"btc-updown-15m-{next_15m}"
pm.get_market(request_session)
print(pm)


fetch_w_aiohttp = datetime.now().timestamp()

# Jupyter already runs an event loop, so await directly instead of asyncio.run()
order_book_yes, order_book_no = await async_get_ob(pm)

after_request = datetime.now().timestamp()

# latest snapshot per direction
set_redis_order_book(order_book_yes)
set_redis_order_book(order_book_no)

# append to history
push_redis_list([order_book_yes, order_book_no])

end = datetime.now().timestamp()

request_session.close()

print(f"fetch with aiohttp: {after_request - fetch_w_aiohttp:0.6f}")
print(f"write to Redis:     {end - after_request:0.6f}")
print(f"total time:         {end - fetch_w_aiohttp:0.6f}")

PolyMarket(slug=btc-updown-15m-1793543400, yes=0.495, no=0.505, status=active, end_date=2026-11-01T14:45:00Z)
fetch with aiohttp: 0.151094
write to Redis:     0.028332
total time:         0.179426
